# Mô hình 1 - SARIMAX đa chân trời t+1 đến t+24

SARIMAX được dùng như baseline chuỗi thời gian. Mô hình được fit riêng cho từng thành
phố với cấu trúc AR(1) và seasonal AR(24). Sau khi ước lượng tham số trên Train, mô hình
thực hiện rolling-origin forecast: tại mỗi origin, trạng thái được cập nhật bằng PM2.5
đã quan sát đến thời điểm đó rồi dự báo đệ quy 24 bước.

Không đưa biến khí tượng tương lai vào SARIMAX vì web hiện không có chuỗi dự báo khí
tượng đồng bộ cho toàn bộ 24 giờ. Điều này tránh vô tình sử dụng dữ liệu tương lai thật.


In [1]:
from pathlib import Path
import json
import random
from typing import Any

import joblib
import matplotlib.pyplot as plt
plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "mathtext.fontset": "dejavusans",
    "axes.unicode_minus": False,
})
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Nhận diện project bằng cấu trúc, không phụ thuộc tên thư mục tạm của project.
CURRENT_DIR = Path.cwd().resolve()
SEARCH_DIRS = [CURRENT_DIR, CURRENT_DIR.parent]
SEARCH_DIRS.extend(path for path in CURRENT_DIR.iterdir() if path.is_dir())
PROJECT_CANDIDATES = []
for candidate in SEARCH_DIRS:
    data_file = candidate / "data" / "processed" / "pm25_training_data_enriched.csv"
    notebook_marker = candidate / "model" / "0_multihorizon_data_preparation.ipynb"
    if notebook_marker.exists() and data_file.exists():
        resolved = candidate.resolve()
        if resolved not in PROJECT_CANDIDATES:
            PROJECT_CANDIDATES.append(resolved)

if len(PROJECT_CANDIDATES) == 1:
    PROJECT_ROOT = PROJECT_CANDIDATES[0]
elif not PROJECT_CANDIDATES:
    raise FileNotFoundError(
        "Không tìm thấy project chứa đồng thời model và "
        "data/processed/pm25_training_data_enriched.csv."
    )
else:
    raise RuntimeError(
        "Có nhiều project phù hợp; hãy mở Jupyter tại đúng thư mục gốc cần chạy: "
        + ", ".join(str(path) for path in PROJECT_CANDIDATES)
    )

MODEL_DIR = PROJECT_ROOT / "model"
RESULTS_DIR = MODEL_DIR / "results"
CANDIDATES_DIR = MODEL_DIR / "candidates"
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "pm25_training_data_enriched.csv"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CANDIDATES_DIR.mkdir(parents=True, exist_ok=True)

HORIZONS = np.arange(1, 25, dtype=int)
TARGET_COLUMNS = [f"target_pm25_t_plus_{h}" for h in HORIZONS]
MAX_HORIZON = int(HORIZONS.max())

POLLUTANT_FEATURES = ["pm25", "pm10", "o3", "no2", "so2", "co"]
WEATHER_FEATURES = [
    "temp", "humidity", "wind_speed", "wind_dir", "precip", "pressure", "cloud_cover"
]
TEMPORAL_FEATURES = ["hour", "day_of_week", "month", "is_weekend", "day_of_year"]
HISTORY_FEATURES = [
    "pm25_lag_1h", "pm25_lag_3h", "pm25_lag_6h", "pm25_lag_12h",
    "pm25_lag_24h", "pm25_lag_48h", "pm25_lag_72h", "pm25_lag_96h",
    "pm25_lag_120h", "pm25_lag_144h", "pm25_lag_168h",
    "pm25_roll_6h", "pm25_roll_12h", "pm25_roll_24h", "pm25_roll_72h",
    "pm25_roll_168h", "pm25_std_6h", "pm25_std_12h", "pm25_std_24h",
    "pm25_std_72h", "pm25_std_168h", "pm25_min_24h", "pm25_max_24h",
    "pm25_delta_1h", "pm25_delta_3h", "pm25_delta_24h",
    "pm25_roll_ratio_6h_24h", "pm25_roll_ratio_24h_72h",
    "pm25_same_hour_mean_7d", "pm25_same_hour_median_7d",
    "pm25_same_hour_std_7d", "pm25_same_hour_min_7d",
    "pm25_same_hour_max_7d", "pm25_same_hour_ratio_7d", "pm25_weekly_delta",
]
ENGINEERED_FEATURES = [
    "ventilation_index", "humid_stagnation", "rain_flag", "calm_wind",
    "high_humidity", "wind_x", "wind_y", "hour_sin", "hour_cos",
    "day_sin", "day_cos", "month_sin", "month_cos", "pm25_pm10_ratio",
    "no2_co_ratio",
]
NUMERIC_FEATURES = (
    POLLUTANT_FEATURES + WEATHER_FEATURES + TEMPORAL_FEATURES
    + HISTORY_FEATURES + ENGINEERED_FEATURES
)
CATEGORICAL_FEATURES = ["city", "season"]

print(f"Project root: {PROJECT_ROOT}")
print(f"Data: {DATA_PATH}")


Project root: D:\Project123456\aqi-vietnam\aqi-vietnam4
Data: D:\Project123456\aqi-vietnam\aqi-vietnam4\data\processed\pm25_training_data_enriched.csv


In [2]:
def make_multihorizon_frame(data: pd.DataFrame) -> pd.DataFrame:
    """Ghép chính xác PM2.5 tại t+1,...,t+24 theo city và timestamp."""
    frame = data.copy()
    frame["datetime"] = pd.to_datetime(frame["datetime"])
    frame = frame.sort_values(["city", "datetime"]).reset_index(drop=True)
    if frame.duplicated(["city", "datetime"]).any():
        raise ValueError("Dữ liệu có city/datetime trùng, không thể ghép target chính xác.")

    lookup = frame.set_index(["city", "datetime"])["pm25"]
    for horizon, column in zip(HORIZONS, TARGET_COLUMNS):
        keys = pd.MultiIndex.from_arrays(
            [frame["city"], frame["datetime"] + pd.to_timedelta(horizon, unit="h")],
            names=["city", "datetime"],
        )
        frame[column] = lookup.reindex(keys).to_numpy(dtype=float)
    frame["forecast_end"] = frame["datetime"] + pd.Timedelta(hours=MAX_HORIZON)
    return frame


def make_temporal_split(
    frame: pd.DataFrame,
    train_fraction: float = 0.70,
    validation_fraction: float = 0.15,
    purge_hours: int = MAX_HORIZON,
) -> dict[str, Any]:
    """Chia theo forecast_end để không có cửa sổ target giao nhau giữa các tập."""
    unique_ends = pd.Series(frame["forecast_end"].dropna().sort_values().unique())
    train_cut = pd.Timestamp(unique_ends.iloc[int(len(unique_ends) * train_fraction)])
    val_cut = pd.Timestamp(
        unique_ends.iloc[int(len(unique_ends) * (train_fraction + validation_fraction))]
    )
    purge = pd.Timedelta(hours=purge_hours)
    forecast_end = pd.to_datetime(frame["forecast_end"])
    masks = {
        "train": forecast_end < train_cut,
        "validation": (forecast_end >= train_cut + purge) & (forecast_end < val_cut),
        "test": forecast_end >= val_cut + purge,
    }
    if any(not mask.any() for mask in masks.values()):
        raise ValueError("Temporal split tạo ra ít nhất một tập rỗng.")
    return {
        "masks": masks,
        "train_cut": train_cut,
        "val_cut": val_cut,
        "purge_hours": purge_hours,
    }


def split_summary(frame: pd.DataFrame, split: dict[str, Any]) -> dict[str, Any]:
    summary: dict[str, Any] = {
        "strategy": "global chronological 70/15/15 by forecast_end with 24-hour purge gaps",
        "train_cut": split["train_cut"].isoformat(),
        "validation_cut": split["val_cut"].isoformat(),
        "purge_hours": int(split["purge_hours"]),
        "horizons": HORIZONS.tolist(),
    }
    for name, mask in split["masks"].items():
        part = frame.loc[mask]
        summary[name] = {
            "rows": int(len(part)),
            "source_start": part["datetime"].min().isoformat(),
            "source_end": part["datetime"].max().isoformat(),
            "forecast_end_start": part["forecast_end"].min().isoformat(),
            "forecast_end_end": part["forecast_end"].max().isoformat(),
        }
    return summary


def load_model_frame() -> tuple[pd.DataFrame, dict[str, Any], pd.DataFrame]:
    raw = pd.read_csv(DATA_PATH, low_memory=False)
    raw["datetime"] = pd.to_datetime(raw["datetime"])
    supervised = make_multihorizon_frame(raw)
    required = NUMERIC_FEATURES + CATEGORICAL_FEATURES + TARGET_COLUMNS
    missing_columns = sorted(set(required) - set(supervised.columns))
    if missing_columns:
        raise KeyError(f"Thiếu cột cần thiết: {missing_columns}")
    frame = supervised.dropna(subset=required).copy().reset_index(drop=True)
    split = make_temporal_split(frame)
    manifest = split_summary(frame, split)
    (RESULTS_DIR / "multihorizon_temporal_split.json").write_text(
        json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    return frame, split, raw


def build_feature_matrix(
    frame: pd.DataFrame,
    feature_columns: list[str] | None = None,
) -> pd.DataFrame:
    matrix = pd.get_dummies(
        frame[NUMERIC_FEATURES + CATEGORICAL_FEATURES],
        columns=CATEGORICAL_FEATURES,
        drop_first=False,
        dtype=float,
    )
    if feature_columns is None:
        return matrix.astype(np.float32)
    for column in feature_columns:
        if column not in matrix:
            matrix[column] = 0.0
    return matrix.reindex(columns=feature_columns, fill_value=0.0).astype(np.float32)


def regression_metrics(actual: Any, predicted: Any) -> dict[str, float]:
    y_true = np.asarray(actual, dtype=float)
    y_pred = np.clip(np.asarray(predicted, dtype=float), 0.0, None)
    return {
        "rmse_ug_m3": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "mae_ug_m3": float(mean_absolute_error(y_true, y_pred)),
        "r2": float(r2_score(y_true, y_pred)),
        "bias_ug_m3": float(np.mean(y_pred - y_true)),
    }


def prediction_frame(
    frame: pd.DataFrame,
    mask: pd.Series,
    predictions: np.ndarray,
    model_name: str,
    split_name: str,
) -> pd.DataFrame:
    part = frame.loc[mask].reset_index(drop=True)
    actual = part[TARGET_COLUMNS].to_numpy(dtype=float)
    predicted = np.clip(np.asarray(predictions, dtype=float), 0.0, None)
    if predicted.shape != actual.shape:
        raise ValueError(f"Prediction shape {predicted.shape} khác target shape {actual.shape}.")
    rows = len(part)
    output = pd.DataFrame({
        "model": model_name,
        "split": split_name,
        "city": np.repeat(part["city"].to_numpy(), len(HORIZONS)),
        "source_time": np.repeat(part["datetime"].to_numpy(), len(HORIZONS)),
        "horizon": np.tile(HORIZONS, rows),
        "actual_pm25": actual.reshape(-1),
        "predicted_pm25": predicted.reshape(-1),
    })
    output["target_time"] = pd.to_datetime(output["source_time"]) + pd.to_timedelta(
        output["horizon"], unit="h"
    )
    output["abs_error_ug_m3"] = np.abs(output["actual_pm25"] - output["predicted_pm25"])
    return output


def metric_tables(predictions: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    horizon_rows = []
    for (model, split_name, horizon), part in predictions.groupby(
        ["model", "split", "horizon"], sort=True
    ):
        horizon_rows.append({
            "model": model,
            "split": split_name,
            "horizon": int(horizon),
            "rows": int(len(part)),
            **regression_metrics(part["actual_pm25"], part["predicted_pm25"]),
        })
    by_horizon = pd.DataFrame(horizon_rows)

    city_rows = []
    for (model, split_name, city), part in predictions.groupby(
        ["model", "split", "city"], sort=True
    ):
        city_rows.append({
            "model": model,
            "split": split_name,
            "city": city,
            "rows": int(len(part)),
            **regression_metrics(part["actual_pm25"], part["predicted_pm25"]),
        })
    by_city = pd.DataFrame(city_rows)

    summary_rows = []
    for (model, split_name), part in predictions.groupby(["model", "split"], sort=True):
        horizon_part = by_horizon.loc[
            by_horizon["model"].eq(model) & by_horizon["split"].eq(split_name)
        ]
        summary_rows.append({
            "model": model,
            "split": split_name,
            "rows": int(len(part)),
            "mean_horizon_rmse_ug_m3": float(horizon_part["rmse_ug_m3"].mean()),
            "mean_horizon_mae_ug_m3": float(horizon_part["mae_ug_m3"].mean()),
            **{f"global_{key}": value for key, value in regression_metrics(
                part["actual_pm25"], part["predicted_pm25"]
            ).items()},
        })
    return by_horizon, by_city, pd.DataFrame(summary_rows)


def save_evaluation(model_slug: str, predictions: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    by_horizon, by_city, summary = metric_tables(predictions)
    predictions.to_csv(
        RESULTS_DIR / f"{model_slug}_multihorizon_predictions.csv",
        index=False,
        encoding="utf-8-sig",
    )
    by_horizon.to_csv(
        RESULTS_DIR / f"{model_slug}_multihorizon_by_horizon.csv",
        index=False,
        encoding="utf-8-sig",
    )
    by_city.to_csv(
        RESULTS_DIR / f"{model_slug}_multihorizon_by_city.csv",
        index=False,
        encoding="utf-8-sig",
    )
    summary.to_csv(
        RESULTS_DIR / f"{model_slug}_multihorizon_summary.csv",
        index=False,
        encoding="utf-8-sig",
    )
    return by_horizon, by_city, summary


In [3]:
import warnings
from statsmodels.tools.sm_exceptions import ConvergenceWarning
from statsmodels.tsa.statespace.sarimax import SARIMAX

SARIMAX_ORDER = (1, 0, 0)
SARIMAX_SEASONAL_ORDER = (1, 0, 0, 24)
MODEL_NAME = "SARIMAX"
MODEL_SLUG = "sarimax"


In [4]:
model_frame, split, raw_data = load_model_frame()
manifest = split_summary(model_frame, split)
display(pd.DataFrame({
    name: values
    for name, values in manifest.items()
    if isinstance(values, dict) and "rows" in values
}).T)


,rows,source_start,source_end,forecast_end_start,forecast_end_end
train,68829,2022-08-12T07:00:00,2025-03-25T05:00:00,2022-08-13T07:00:00,2025-03-26T05:00:00
validation,14679,2025-03-26T06:00:00,2025-10-16T02:00:00,2025-03-27T06:00:00,2025-10-17T02:00:00
test,14679,2025-10-17T03:00:00,2026-05-08T23:00:00,2025-10-18T03:00:00,2026-05-09T23:00:00


## Hàm dự báo đệ quy

Với SARIMAX AR(1) x SAR(1,24), phương trình rút gọn chứa các độ trễ 1, 24 và 25.
Hàm dưới đây sử dụng giá trị quan sát cho các thời điểm không vượt quá origin và sử
dụng chính dự báo trước đó cho các bước tương lai. Đây là dự báo multi-step thực sự,
không phải sao chép giá trị hiện tại.


In [5]:
def recursive_sarimax_forecast(
    history: pd.Series,
    origins: pd.Series,
    intercept: float,
    ar1: float,
    seasonal_ar24: float,
) -> np.ndarray:
    values = history.to_numpy(dtype=float)
    positions = history.index.get_indexer(pd.DatetimeIndex(origins))
    if (positions < 0).any():
        raise ValueError("Có origin không tồn tại trong chuỗi SARIMAX.")

    output = np.empty((len(origins), len(HORIZONS)), dtype=float)
    for row, origin_position in enumerate(positions):
        forecasts: list[float] = []

        def lag_value(relative_position: int) -> float:
            if relative_position <= 0:
                return float(values[origin_position + relative_position])
            return float(forecasts[relative_position - 1])

        for horizon in HORIZONS:
            value = (
                intercept
                + ar1 * lag_value(int(horizon) - 1)
                + seasonal_ar24 * lag_value(int(horizon) - 24)
                - ar1 * seasonal_ar24 * lag_value(int(horizon) - 25)
            )
            forecasts.append(max(0.0, float(value)))
        output[row] = forecasts
    return output


In [6]:
city_artifacts: dict[str, Any] = {}
prediction_parts = []
fit_rows = []

for city_number, city in enumerate(sorted(model_frame["city"].unique()), start=1):
    city_origins = model_frame.loc[model_frame["city"].eq(city)].copy()
    city_history = (
        raw_data.loc[raw_data["city"].eq(city), ["datetime", "pm25"]]
        .drop_duplicates("datetime")
        .sort_values("datetime")
        .set_index("datetime")["pm25"]
        .asfreq("h")
        .interpolate(method="time", limit=6, limit_area="inside")
        .ffill()
        .bfill()
    )
    train_origins = city_origins.loc[split["masks"]["train"].loc[city_origins.index]]
    fit_end = train_origins["datetime"].max()
    fit_series = city_history.loc[:fit_end].astype(float)

    print(f"[SARIMAX {city_number}/3] Đang fit {city}: {len(fit_series):,} giờ")
    with warnings.catch_warnings():
        warnings.simplefilter("always", ConvergenceWarning)
        model = SARIMAX(
            fit_series,
            order=SARIMAX_ORDER,
            seasonal_order=SARIMAX_SEASONAL_ORDER,
            trend="c",
            enforce_stationarity=False,
            enforce_invertibility=False,
        )
        result = model.fit(method="lbfgs", maxiter=200, disp=False)

    converged = bool(result.mle_retvals.get("converged", False))
    params = result.params
    intercept = float(params.get("intercept", 0.0))
    ar1 = float(params["ar.L1"])
    seasonal_ar24 = float(params["ar.S.L24"])
    city_artifacts[city] = {
        "intercept": intercept,
        "ar_l1": ar1,
        "seasonal_ar_l24": seasonal_ar24,
        "aic": float(result.aic),
        "converged": converged,
        "fit_end": fit_end.isoformat(),
    }
    fit_rows.append({"city": city, **city_artifacts[city]})
    print(f"[SARIMAX {city_number}/3] Hoàn tất: converged={converged}, AIC={result.aic:.2f}")

    for split_name in ("validation", "test"):
        city_mask = split["masks"][split_name] & model_frame["city"].eq(city)
        part = model_frame.loc[city_mask]
        predicted = recursive_sarimax_forecast(
            city_history,
            part["datetime"],
            intercept,
            ar1,
            seasonal_ar24,
        )
        prediction_parts.append(
            prediction_frame(model_frame, city_mask, predicted, MODEL_NAME, split_name)
        )

predictions = pd.concat(prediction_parts, ignore_index=True)
fit_report = pd.DataFrame(fit_rows)
display(fit_report)


[SARIMAX 1/3] Đang fit Hà Nội: 23,111 giờ


[SARIMAX 1/3] Hoàn tất: converged=True, AIC=144725.93


[SARIMAX 2/3] Đang fit TP.HCM: 23,111 giờ


[SARIMAX 2/3] Hoàn tất: converged=True, AIC=125290.64


[SARIMAX 3/3] Đang fit Đà Nẵng: 23,111 giờ


[SARIMAX 3/3] Hoàn tất: converged=True, AIC=104045.23


,city,intercept,ar_l1,seasonal_ar_l24,aic,converged,fit_end
0,Hà Nội,0.838775,0.967801,0.389764,144725.931753,True,2025-03-25T05:00:00
1,TP.HCM,0.944257,0.940102,0.359657,125290.642167,True,2025-03-25T05:00:00
2,Đà Nẵng,0.496521,0.957379,0.391905,104045.232874,True,2025-03-25T05:00:00


In [7]:
artifact = {
    "model": MODEL_NAME,
    "strategy": "city-specific rolling-origin recursive 24-step forecast",
    "order": SARIMAX_ORDER,
    "seasonal_order": SARIMAX_SEASONAL_ORDER,
    "horizons": HORIZONS.tolist(),
    "city_parameters": city_artifacts,
}
joblib.dump(artifact, CANDIDATES_DIR / "sarimax_multihorizon.joblib")
fit_report.to_csv(
    RESULTS_DIR / "sarimax_fit_report.csv", index=False, encoding="utf-8-sig"
)
by_horizon, by_city, summary = save_evaluation(MODEL_SLUG, predictions)
display(summary)
display(by_city)


,model,split,rows,mean_horizon_rmse_ug_m3,mean_horizon_mae_ug_m3,global_rmse_ug_m3,global_mae_ug_m3,global_r2,global_bias_ug_m3
0,SARIMAX,test,352296,16.007236,9.847911,16.367168,9.847911,0.566209,-0.546474
1,SARIMAX,validation,352296,17.764182,10.989859,18.185878,10.989859,0.476800,-1.465380


,model,split,city,rows,rmse_ug_m3,mae_ug_m3,r2,bias_ug_m3
0,SARIMAX,test,Hà Nội,117432,25.353868,17.719819,0.359369,-1.557154
1,SARIMAX,test,TP.HCM,117432,10.600093,7.265787,0.287180,-0.383360
2,SARIMAX,test,Đà Nẵng,117432,6.962183,4.558127,0.526574,0.301093
3,SARIMAX,validation,Hà Nội,117432,26.154597,17.421226,0.227715,-1.702576
4,SARIMAX,validation,TP.HCM,117432,14.734987,9.855388,0.333500,-2.599190
5,SARIMAX,validation,Đà Nẵng,117432,9.539168,5.692961,0.480027,-0.094373


In [8]:
test_curve = by_horizon.loc[by_horizon["split"].eq("test")]
axis = test_curve.plot(
    x="horizon", y=["rmse_ug_m3", "mae_ug_m3"], marker="o", figsize=(9, 4.5)
)
axis.set_xlabel("Chân trời dự báo (giờ)")
axis.set_ylabel("Sai số (µg/m³)")
axis.set_title("SARIMAX: sai số Test theo chân trời")
axis.grid(alpha=0.25)
plt.tight_layout()
plt.show()


C:\Users\nguyen\AppData\Local\Temp\ipykernel_2388\2843656106.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [9]:
non_converged = fit_report.loc[~fit_report["converged"], "city"].tolist()
test_summary = summary.loc[summary["split"].eq("test")].iloc[0]
convergence_text = (
    "Tất cả mô hình theo thành phố đã hội tụ."
    if not non_converged
    else f"Chưa hội tụ tại: {', '.join(non_converged)}; kết quả này cần được xem là baseline thận trọng."
)
display(Markdown(
    f"**Nhận xét.** {convergence_text} RMSE Test trung bình theo 24 horizon là "
    f"**{test_summary['mean_horizon_rmse_ug_m3']:.2f} µg/m³** và MAE là "
    f"**{test_summary['mean_horizon_mae_ug_m3']:.2f} µg/m³**. Sai số theo horizon "
    "cho biết tốc độ suy giảm độ chính xác khi khoảng dự báo dài hơn."
))


**Nhận xét.** Tất cả mô hình theo thành phố đã hội tụ. RMSE Test trung bình theo 24 horizon là **16.01 µg/m³** và MAE là **9.85 µg/m³**. Sai số theo horizon cho biết tốc độ suy giảm độ chính xác khi khoảng dự báo dài hơn.